# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedsamymohamad/flyrank_internship_starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring**

Four things in order:
1. The contract — five plain-words answers
2. Three verification queries with visible outputs (on `month=2026-03`)
3. Five features with "knowable when?" lines, plus the deliberate-leak experiment
4. One named limitation

> ⚠️ **Before running:** request gate access at https://huggingface.co/datasets/FlyRank/internship-warehouse (instant approval), create a plain **Read** token in your HF settings, and store it as a Colab Secret named `HF_TOKEN`. Never paste the token into a cell — this repo is public.

---
## 0. Setup — install, authenticate, load

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'datasets', 'huggingface_hub'],
    check=True
)
print('Dependencies ready.')

In [ ]:
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    raise EnvironmentError(
        'HF_TOKEN not found. Add it as a Colab Secret named HF_TOKEN '
        '(the key icon in the left sidebar). Never paste the token into a cell.'
    )
print('Token loaded (length:', len(HF_TOKEN), ')')

In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset

REPO   = "FlyRank/internship-warehouse"
# Development month — mid-panel, safe for label logic.
# NEVER use month=2026-06 (_sample) for label development: it is the sealed test month.
DEV_MONTH = '2026-03'

# ── 1. Load the daily fact table (streaming — no full download) ───────────────
print("Loading fact_content_daily_performance (streaming)...")
ds_fact = load_dataset(
    REPO,
    "fact_content_daily_performance",
    streaming=True,
    split="train",
    token=HF_TOKEN,
)

# Filter to the development month.
# report_date may arrive as a date object or string; str()[:7] handles both.
ds_march = ds_fact.filter(
    lambda row: str(row['report_date'])[:7] == DEV_MONTH
)

# Collect into a DataFrame.
# The filter still streams the full table — this takes a few minutes on Colab.
# We cap at 500 000 rows for development; cache the result below.
print(f"Collecting month={DEV_MONTH} rows (cap 500 000)...")
df_fact = pd.DataFrame(list(ds_march.take(500_000)))
print(f"fact rows collected : {len(df_fact):,}")
print(f"fact columns        : {list(df_fact.columns)}")

In [ ]:
# ── 2. Load fact_content_query_90d (single flat file — faster) ───────────────
print("Loading fact_content_query_90d (streaming)...")
ds_q90d = load_dataset(
    REPO,
    "fact_content_query_90d",
    streaming=True,
    split="train",
    token=HF_TOKEN,
)

# The query table's window is fixed (the most recent ~90 days of the snapshot).
# We cap at 200 000 rows — enough for a representative feature frame.
print("Collecting query-90d rows (cap 200 000)...")
df_q90d = pd.DataFrame(list(ds_q90d.take(200_000)))
print(f"q90d rows collected : {len(df_q90d):,}")
print(f"q90d columns        : {list(df_q90d.columns)}")

In [ ]:
# ── Optional: cache to work/outputs/ so you only stream once ─────────────────
import pathlib
out = pathlib.Path('../../work/outputs')
out.mkdir(parents=True, exist_ok=True)

df_fact.to_parquet(out / f'fact_march2026_sample.parquet', index=False)
df_q90d.to_parquet(out / f'q90d_sample.parquet', index=False)
print('Cached to work/outputs/  (reload from there to skip streaming next time)')

---
## 1. The Contract — five plain-words answers

### 1.1 — One row means…

One row = **one content item (page) on one calendar date for one client**.
The grain of `fact_content_daily_performance` is `(report_date, client_hash_id, content_hash_id)` — a single day's measured performance for a single page belonging to a single client.

### 1.2 — Table(s) I'll use

**Primary:** `fact_content_daily_performance` (config name in `load_dataset`). I develop on `month=2026-03` (a mid-panel month). I treat `month=2026-06` (the final month / `_sample`) as a **sealed test month** — developing label logic there means developing inside the natural outcome window of any past→future label.

**For 30-day window features:** `fact_content_query_90d` — a single flat file pre-computing `impressions_prev30`, `clicks_prev30`, `avg_position_prev30` (days 31–60 back) and `impressions_last30`, `clicks_last30` (most recent 30 days). These are safe feature and label sources when windows are aligned correctly.

### 1.3 — Time window

The **feature window** is the prev-30 sub-window (days 31–60 before snapshot end): `impressions_prev30`, `clicks_prev30`, `avg_position_prev30`. The **label window** is the last-30 sub-window. Decline is detected when `impressions_last30 < impressions_prev30 × 0.80`. The two windows do not overlap — that boundary is what the leakage experiment deliberately crosses.

### 1.4 — What I predict (label / proxy)

**Proxy label:** `is_declining = 1` when impressions in the most-recent 30 days fell more than 20% compared with the prior 30 days. This is a **rule-based proxy** — not a directly observed editorial outcome. Results are labelled as directional / decision-support, not causal.

### 1.5 — One deliberate exclusion

I exclude **`impressions_last30`** and any column derived from the last-30d window (`clicks_last30`, `avg_position_last30`). These overlap with the label's computation window — using them as features means the model reads the answer. Section 3 demonstrates this deliberately.

In [ ]:
contract = {
    '1 — One row':  'one content item on one report_date for one client '
                    '→ grain: (report_date, client_hash_id, content_hash_id)',
    '2 — Tables':   'fact_content_daily_performance (month=2026-03, streamed); '
                    'fact_content_query_90d (streamed) for pre-aggregated 30d window features',
    '3 — Window':   'features from prev30 window (days 31-60 back); '
                    'label from last30 vs prev30 impressions comparison',
    '4 — Label':    'is_declining = 1 when impressions_last30 / impressions_prev30 < 0.80 '
                    '(rule-based proxy; not a directly observed editorial outcome)',
    '5 — Excluded': 'impressions_last30, clicks_last30, avg_position_last30 '
                    '(overlap label window → leakage); '
                    'client_hash_id, content_hash_id (IDs — grouping/joins only, never features)',
}

print('=== DATA CONTRACT — Lane 2: Refresh / Content Opportunity Scoring ===')
for k, v in contract.items():
    print(f'\n{k}:\n  {v}')

---
## 2. Verify it — three checks with visible outputs

Same three facts as before, now verified in pandas on the streamed DataFrames.

In [ ]:
# ── Check 1: GRAIN ────────────────────────────────────────────────────────────
# Claim: one row = one (report_date, client_hash_id, content_hash_id).
# Zero duplicate triples → grain holds.

grain_cols = ['report_date', 'client_hash_id', 'content_hash_id']
dupes = df_fact[df_fact.duplicated(subset=grain_cols, keep=False)]

print('Check 1 — Grain: rows where (report_date, client_hash_id, content_hash_id) appears > 1 time')
print(f'Duplicate rows : {len(dupes)}  (0 = grain holds ✓)')
if len(dupes) > 0:
    print(dupes[grain_cols + ['gsc_impressions']].head())

In [ ]:
# ── Check 2: ROW COUNT + DATE SPAN ────────────────────────────────────────────
# Confirm shape and date range of the collected month=2026-03 slice.

df_fact['report_date'] = pd.to_datetime(df_fact['report_date'])

print('Check 2 — Row count and date span for month=2026-03')
print(f"  total_rows      : {len(df_fact):,}")
print(f"  unique_clients  : {df_fact['client_hash_id'].nunique():,}")
print(f"  unique_pages    : {df_fact['content_hash_id'].nunique():,}")
print(f"  earliest_date   : {df_fact['report_date'].min().date()}")
print(f"  latest_date     : {df_fact['report_date'].max().date()}")

In [ ]:
# ── Check 3: AVAILABILITY — IS TRUE logic ─────────────────────────────────────
# ga4_data_available / gsc_data_available are THREE-valued: True, False, or None/NaN.
# Python `== True` silently treats None as False — same bug as SQL `= TRUE`.
# The correct equivalent of SQL `IS TRUE` in pandas is: col.eq(True) (NaN → False).

total          = len(df_fact)
gsc_true       = df_fact['gsc_data_available'].eq(True).sum()
ga4_true       = df_fact['ga4_data_available'].eq(True).sum()
both_true      = (df_fact['gsc_data_available'].eq(True) &
                  df_fact['ga4_data_available'].eq(True)).sum()

print('Check 3 — Availability: IS TRUE filter on gsc_data_available and ga4_data_available')
print(f"  total_rows             : {total:,}")
print(f"  gsc_available_rows     : {gsc_true:,}  ({gsc_true/total:.1%})")
print(f"  ga4_available_rows     : {ga4_true:,}  ({ga4_true/total:.1%})")
print(f"  both_available_rows    : {both_true:,}  ({both_true/total:.1%})")
print()
print('Note: .eq(True) is the pandas equivalent of SQL IS TRUE.')
print('It returns False for both False and NaN — correctly excluding null-metric rows.')

---
## 3. Five features + the leakage trap

### 3a. Feature frame

All five features are from the **prev30 window** (days 31–60 before snapshot) or aggregated from the daily fact. None overlaps with the label window (last30).

| Feature | Source column | Available when? |
|---|---|---|
| `log_impressions_prev30` | `log1p(impressions_prev30)` from `fact_content_query_90d` | Knowable at decision moment — covers days 31–60 before snapshot, entirely before the last30 label window |
| `log_clicks_prev30` | `log1p(clicks_prev30)` from `fact_content_query_90d` | Knowable at decision moment — same prev30 window, no overlap with label |
| `avg_position_prev30` | `avg_position_prev30` from `fact_content_query_90d` | Knowable at decision moment — average GSC rank in days 31–60 back; not derived from the label window |
| `days_with_gsc` | `COUNT(report_date WHERE gsc_impressions > 0)` from daily fact | Knowable at decision moment — days in the month with any impression; measures visibility regularity |
| `gsc_avg_position_month` | `mean(gsc_avg_position WHERE > 0)` from daily fact | Knowable at decision moment — average daily position over the whole month; not a last30 column |

In [ ]:
# ── Features from fact_content_query_90d ─────────────────────────────────────
# The query table grain is (client_hash_id, content_hash_id, query_hash_id).
# Per-content context columns repeat on every query row — aggregate with first().

agg = df_q90d.groupby(
    ['client_hash_id', 'content_hash_id'], as_index=False
).agg(
    impressions_prev30   = ('impressions_prev30', 'first'),
    clicks_prev30        = ('clicks_prev30',      'first'),
    avg_position_prev30  = ('avg_position_prev30','first'),
    # Keep label inputs separate — NOT features
    impressions_last30   = ('impressions_last30',  'first'),
)

# Only keep rows with a non-zero prev30 baseline (needed to compute the label)
agg = agg[agg['impressions_prev30'] > 0].copy()

# Safe features (prev30 window)
agg['log_impressions_prev30'] = np.log1p(agg['impressions_prev30'])
agg['log_clicks_prev30']      = np.log1p(agg['clicks_prev30'])

# Proxy label
agg['is_declining'] = (
    agg['impressions_last30'] < agg['impressions_prev30'] * 0.80
).astype(int)

print(f'q90d feature frame shape : {agg.shape}')
print(f'Label rate (is_declining=1): {agg["is_declining"].mean():.1%}')
agg.head(3)

In [ ]:
# ── Two more features from the daily fact (month=2026-03) ────────────────────
# Restrict to rows where GSC data is actually available (IS TRUE equivalent)
df_gsc = df_fact[df_fact['gsc_data_available'].eq(True)].copy()

# gsc_avg_position = 0 means "no position data that day" — exclude from mean
df_gsc['position_clean'] = df_gsc['gsc_avg_position'].where(df_gsc['gsc_avg_position'] > 0)

daily_feats = df_gsc.groupby(
    ['client_hash_id', 'content_hash_id'], as_index=False
).agg(
    days_with_gsc          = ('gsc_impressions',  lambda x: (x > 0).sum()),
    gsc_avg_position_month = ('position_clean',   'mean'),
)

print(f'Daily feature frame shape : {daily_feats.shape}')
daily_feats.head(3)

In [ ]:
# ── Join into the final feature frame ────────────────────────────────────────
feat = agg.merge(
    daily_feats,
    on=['client_hash_id', 'content_hash_id'],
    how='inner'
).dropna(subset=['log_impressions_prev30', 'log_clicks_prev30',
                 'avg_position_prev30', 'days_with_gsc'])

feature_cols = [
    'log_impressions_prev30',
    'log_clicks_prev30',
    'avg_position_prev30',
    'days_with_gsc',
    'gsc_avg_position_month',
]

print(f'Final feature frame shape : {feat.shape}')
print(f'Label rate (is_declining=1): {feat["is_declining"].mean():.1%}')
print()
print('Feature summary:')
feat[feature_cols].describe().round(3)

### 3b. The leakage trap — watch the score jump, then delete the leak

We deliberately add `impressions_last30` as a 6th feature — the numerator of the declining label. AUC jumps toward perfect. Then we remove it and report the honest number.

> **The lesson from notebook 02, performed on real warehouse data:** any column that overlaps with the label's computation window makes the model look nearly perfect. That performance is not real — the model is reading the answer. Delete it before a single line of model code runs.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Add the leaked column: log1p(impressions_last30) — the label's numerator
feat = feat.copy()
feat['log_impressions_last30_LEAK'] = np.log1p(feat['impressions_last30'])

y   = feat['is_declining']
clf = Pipeline([('scaler', StandardScaler()),
                ('lr', LogisticRegression(max_iter=1000))])

# Score WITH the leak
X_leaked   = feat[feature_cols + ['log_impressions_last30_LEAK']]
auc_leaked = cross_val_score(clf, X_leaked, y, cv=5, scoring='roc_auc').mean()
print(f'[LEAKED  ] ROC-AUC with impressions_last30 included : {auc_leaked:.4f}  ← suspiciously high')

# Score WITHOUT the leak (honest)
X_honest   = feat[feature_cols]
auc_honest = cross_val_score(clf, X_honest, y, cv=5, scoring='roc_auc').mean()
print(f'[HONEST  ] ROC-AUC after removing the leak         : {auc_honest:.4f}  ← real signal')
print()
print(f'AUC gap caused by leakage : {auc_leaked - auc_honest:+.4f}')
print()
print('Conclusion: log_impressions_last30_LEAK is DELETED.')
print('The honest ROC-AUC above is the number we carry forward.')

# Remove leaked and raw label-input columns — must not appear downstream
feat.drop(
    columns=['log_impressions_last30_LEAK', 'impressions_last30',
             'impressions_prev30', 'clicks_prev30'],
    inplace=True
)

In [ ]:
# Confirm the final feature set — no label-window columns remain
print('Final feature set (safe — no label-window overlap):')
for f in feature_cols:
    print(f'  ✓  {f}')
print()
print('Excluded (leakage or IDs):')
for e in ['impressions_last30', 'clicks_last30', 'avg_position_last30',
           'client_hash_id', 'content_hash_id']:
    print(f'  ✗  {e}')
print()
print('Remaining columns in feat :', list(feat.columns))

---
## 4. Data limits — one named limitation

**Unbalanced panel depth makes cross-client normalisation necessary.**

The `fact_content_daily_performance` panel is unbalanced: per-client history depth ranges from roughly 3 to 17 months depending on each client's `gsc_data_start` (stored in `dim_clients`). In `month=2026-03`, clients who joined the platform after mid-2025 may have fewer than 30 days of measurable prior-period history — meaning their `impressions_prev30` reflects a shorter baseline rather than a true 30-day trailing window. A model trained on raw cross-client values will conflate "new client, limited history" with "declining page". Any cross-client model must either normalise per-client, use per-client baseline ratios, or filter to clients with `gsc_data_start ≤ 2025-08` before defining the label. This limitation is invisible in the per-row data; it requires a join to `dim_clients`.

In [ ]:
# Illustrate: distinct days per client in the collected month=2026-03 slice.
# A full month should yield ~31 distinct days. Clients below that have short history.

panel = (
    df_fact
    .groupby('client_hash_id', as_index=False)
    .agg(
        distinct_days      = ('report_date', 'nunique'),
        gsc_available_rows = ('gsc_data_available', lambda x: x.eq(True).sum()),
        ga4_available_rows = ('ga4_data_available', lambda x: x.eq(True).sum()),
    )
    .sort_values('distinct_days')
)

print('Ten clients with fewest distinct days in month=2026-03 (bottom of the unbalanced panel):')
print(panel.head(10).to_string(index=False))
print()
print('Limitation confirmed: clients with < 31 distinct days have short or uneven history.')
print('Per-client normalisation is required before cross-client modelling.')

---
## 5. Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Three verification checks with visible outputs — grain (0 dupes), counts+dates, IS TRUE availability
- [x] Five features, each with an "available when?" line in the table above
- [x] The deliberate-leak experiment is shown, the AUC gap is quantified, the column is deleted, honest number kept
- [x] One named limitation stated in plain words and illustrated with a query
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.